In [ ]:
import torch
from itertools import product

num_modes = 12
d_v = 64
x = torch.randn((d_v, 256, 256))

# Fourier transform
X = torch.fft.rfft2(x) # Outputfut of F(v_t)
X = X[:, :num_modes, :num_modes].permute(*torch.arange(x.ndim - 1, -1, -1))

R = torch.nn.Parameter(torch.randn(num_modes, num_modes, d_v, d_v)).to(torch.complex64)


In [155]:
x = torch.randn(1, 5, 5)
k = torch.fft.fft2(x)[0].numpy()
import numpy as np
# Format each complex number nicely
def fmt(c):
    return f"{c.real:+7.3f}{c.imag:+7.3f}j"

# Print with row and column labels to see Hermitian symmetry
print("Hermitian FFT2 symmetry: v[k1,k2] = conj(v[-k1,-k2])")
print("For a 5×5 grid, freq indices wrap: -1↔4, -2↔3")
print()
print("        " + "   ".join(f"k2={j}" for j in range(5)))
for i in range(5):
    row_str = "   ".join(fmt(k[i, j]) for j in range(5))
    print(f"k1={i}  {row_str}")

print("\n" + "="*80)
print("Checking Hermitian conjugate pairs (should all be True):")
print(f"  k[0,1] ≈ conj(k[0,4])?  {np.allclose(k[0,1], np.conj(k[0,4]))}")
print(f"  k[1,1] ≈ conj(k[4,4])?  {np.allclose(k[1,1], np.conj(k[4,4]))}")
print(f"  k[1,2] ≈ conj(k[4,3])?  {np.allclose(k[1,2], np.conj(k[4,3]))}")
print(f"  k[2,0] ≈ conj(k[3,0])?  {np.allclose(k[2,0], np.conj(k[3,0]))}")

Hermitian FFT2 symmetry: v[k1,k2] = conj(v[-k1,-k2])
For a 5×5 grid, freq indices wrap: -1↔4, -2↔3

        k2=0   k2=1   k2=2   k2=3   k2=4
k1=0   -1.636 +0.000j    +0.870 +2.233j    +5.003 +1.163j    +5.003 -1.163j    +0.870 -2.233j
k1=1   +5.489 -2.769j    +1.240 -2.285j    +1.105 +5.320j    -1.364 +0.582j    +3.074 -0.040j
k1=2   -0.138 -6.420j    -0.743 -2.614j    -0.894 -1.337j    -4.473 +2.083j    +0.564 +3.235j
k1=3   -0.138 +6.420j    +0.564 -3.235j    -4.473 -2.083j    -0.894 +1.337j    -0.743 +2.614j
k1=4   +5.489 +2.769j    +3.074 +0.040j    -1.364 -0.582j    +1.105 -5.320j    +1.240 +2.285j

Checking Hermitian conjugate pairs (should all be True):
  k[0,1] ≈ conj(k[0,4])?  True
  k[1,1] ≈ conj(k[4,4])?  True
  k[1,2] ≈ conj(k[4,3])?  True
  k[2,0] ≈ conj(k[3,0])?  True


In [145]:
X.shape

torch.Size([12, 12, 64])

In [72]:
Y = torch.einsum('okij, okj -> oki', R, X)

In [95]:
k_max = 12
dv = 64
Fv = torch.randn((k_max, dv))
R = torch.randn((k_max, dv, dv))

In [ ]:
def apply_linear_simple(Fv, R):
    out = torch.empty_like(Fv)
    k = Fv.shape[0]
    l = Fv.shape[-1]

    for k_idx in range(k):
        for l_idx in range(l):
            out[k_idx, l_idx] = R[k_idx,l_idx, :] @ Fv[k_idx, :]

    return out
test = apply_linear_simple(Fv, R)
test_ein = torch.einsum('klj, kj -> kl', R, Fv)

print(Fv.shape, Fv.dtype)
print(R.shape, R.dtype)

a = apply_linear_simple(Fv, R)
b = torch.einsum('klm,km->kl', R, Fv)
print(a.shape, b.shape)
print((a - b).abs().max(), torch.allclose(a, b, atol=1e-5))

In [144]:
Fv = torch.randn((k_max, k_max, dv))
R = torch.randn((k_max, k_max, dv, dv))
def apply_linear_complicated(Fv, R):
    out = torch.empty_like(Fv)
    k = Fv.shape[0]
    l = R.shape[2]
    
    for k_tup in list(product(range(k), repeat=2)):
        k_idx1, k_idx2 = k_tup

        for l_idx in range(l):
            out[k_idx1, k_idx2, l_idx] = R[k_idx1, k_idx2, l_idx, :] @ Fv[k_idx1, k_idx2, :]

    return out

print(Fv.shape, Fv.dtype)
print(R.shape, R.dtype)

a = apply_linear_complicated(Fv, R)
b = torch.einsum('kxlj,kxj->kxl', R, Fv)
print(a.shape, b.shape)
print((a - b).abs().max(), torch.allclose(a, b, atol=1e-5))

torch.Size([12, 12, 64]) torch.float32
torch.Size([12, 12, 64, 64]) torch.float32
torch.Size([12, 12, 64]) torch.Size([12, 12, 64])
tensor(3.8147e-06) True


In [149]:
torch.fft.ifft2(b).shape

torch.Size([12, 12, 64])